In [ ]:
"""
================================================================================
BirdCLEF+ 2026 - v8 Inference
================================================================================

Pipeline: Perch -> ProtoSSM(pass1) + MLP -> ensemble -> ResidualSSM(pass2) -> final

Competition Constraints:
  - CPU Notebook <= 90 minutes run-time
  - GPU submissions disabled (only 1 minute)
  - Internet access disabled
  - External data allowed (pre-trained models)
  - Submission file must be named submission.csv

Required Input Files (from training notebook):
  - full_perch_arrays.npz    → models + preprocessing + indices
  - full_perch_meta.parquet  → metadata

Output:
  - submission.csv           → predictions for test soundscapes

Time Budget Analysis (for ~1100 test files):
  - Perch inference:         ~45-55 min (dominant)
  - ProtoSSM forward:        ~5 sec
  - MLP probes:              ~30 sec
  - ResidualSSM forward:     ~5 sec
  - Other operations:        ~2 min
  - Total:                   ~50-60 min (within 90 min limit)

================================================================================
"""

# =============================================================================
# INSTALL TENSORFLOW 2.20 (Required for Perch v2)
# =============================================================================
!pip install -q --no-deps /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl
!pip install -q --no-deps /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc, json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from tqdm.auto import tqdm
import pickle
import io

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()
# =============================================================================
# CONFIG
# =============================================================================
BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
ARTIFACTS_DIR = Path("/kaggle/input/notebooks/blamerx/birdclef-2026-training")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12
BATCH_FILES = 16
DRYRUN_N_FILES = 20

DEVICE = torch.device("cpu")

print("=" * 60)
print("BirdCLEF+ 2026 - v8 Inference")
print("=" * 60)

# =============================================================================
# LOAD ARTIFACTS
# =============================================================================
print("\nLoading artifacts...")

arr = np.load(ARTIFACTS_DIR / "full_perch_arrays.npz", allow_pickle=True)

# Labels
PRIMARY_LABELS = arr["labels"].tolist() if "labels" in arr.files else arr["primary_labels"].tolist()
N_CLASSES = len(PRIMARY_LABELS)

# Config
config = json.loads(str(arr["config"])) if "config" in arr.files else {}
ENSEMBLE_WEIGHT = float(config.get("ensemble_weight", 0.0))
CORRECTION_WEIGHT = float(config.get("correction_weight", 0.3))
BEST_FUSION = config.get("best_fusion", {
    "lambda_event": 0.4,
    "lambda_texture": 1.0,
    "lambda_proxy_texture": 0.8,
    "smooth_texture": 0.35,
    "smooth_event": 0.15,
})
PROBE_ALPHA = config.get("frozen_best_probe", {}).get("alpha", 0.4)
TEMPERATURE = config.get("frozen_best_probe", {}).get("temperature", 1.10)

print(f"Classes: {N_CLASSES}")
print(f"Ensemble weight (ProtoSSM): {ENSEMBLE_WEIGHT}")
print(f"Correction weight: {CORRECTION_WEIGHT}")
print(f"Probe alpha: {PROBE_ALPHA}")
print(f"Temperature: {TEMPERATURE}")

# Index arrays
BC_INDICES = arr["bc_indices"]
MAPPED_POS = arr["mapped_pos"]
MAPPED_BC_INDICES = arr["mapped_bc_indices"]
selected_proxy_pos = arr["selected_proxy_pos"]
idx_active_texture = arr["idx_active_texture"]
idx_active_event = arr["idx_active_event"]
idx_mapped_active_texture = arr["idx_mapped_active_texture"]
idx_mapped_active_event = arr["idx_mapped_active_event"]
idx_selected_proxy_active_texture = arr["idx_selected_proxy_active_texture"]
idx_selected_prioronly_active_texture = arr["idx_selected_prioronly_active_texture"]
idx_selected_prioronly_active_event = arr["idx_selected_prioronly_active_event"]
idx_unmapped_inactive = arr["idx_unmapped_inactive"]

# Proxy map
proxy_keys = arr["proxy_map_keys"]
proxy_vals = arr["proxy_map_vals"]
selected_proxy_pos_to_bc = {int(k): np.array(v, dtype=np.int32) for k, v in zip(proxy_keys, proxy_vals)}

# Deserialize models from bytes
def deserialize_pickle(arr_bytes):
    buf = io.BytesIO(arr_bytes.tobytes())
    return pickle.load(buf)

# Load probes
mlp_probes = deserialize_pickle(arr["mlp_probes_bytes"]) if "mlp_probes_bytes" in arr.files else {}
lr_probes = deserialize_pickle(arr["lr_probes_bytes"]) if "lr_probes_bytes" in arr.files else {}

# Preprocessing - handle both formats
if "emb_scaler_bytes" in arr.files:
    emb_scaler = deserialize_pickle(arr["emb_scaler_bytes"])
    emb_pca = deserialize_pickle(arr["emb_pca_bytes"])
else:
    # Use numpy arrays directly
    scaler_mean = arr["scaler_mean"]
    scaler_scale = arr["scaler_scale"]
    pca_components = arr["pca_components"]
    pca_mean = arr["pca_mean"]
    emb_scaler = None
    emb_pca = None

# Prior tables
if "prior_tables_bytes" in arr.files:
    prior_tables = deserialize_pickle(arr["prior_tables_bytes"])
else:
    site_to_i = dict(arr["site_to_i"]) if "site_to_i" in arr.files else {}
    hour_to_i = dict(arr["hour_to_i"]) if "hour_to_i" in arr.files else {}
    prior_tables = {
        "global_p": arr["global_p"],
        "site_to_i": site_to_i,
        "site_n": arr["site_n"],
        "site_p": arr["site_p"],
        "hour_to_i": hour_to_i,
        "hour_n": arr["hour_n"],
        "hour_p": arr["hour_p"],
        "sh_p": arr.get("sh_p", np.zeros((0, N_CLASSES), dtype=np.float32)),
        "sh_n": arr.get("sh_n", np.array([], dtype=np.float32)),
        "sh_to_i": {},
    }

print(f"MLP probes: {len(mlp_probes)}, LR probes: {len(lr_probes)}")

# =============================================================================
# MODEL DEFINITIONS (for loading PyTorch models)
# =============================================================================
class SelectiveSSM(nn.Module):
    """Simplified Mamba-style selective state space model."""

    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)

        self.conv1d = nn.Conv1d(
            d_model, d_model, d_conv,
            padding=d_conv - 1, groups=d_model
        )

        self.dt_proj = nn.Linear(d_model, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        A = A.unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))

        self.D = nn.Parameter(torch.ones(d_model))

        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_size, T, D = x.shape

        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))
        B_t = self.B_proj(x_conv)
        C_t = self.C_proj(x_conv)
        A = -torch.exp(self.A_log)

        y = self._selective_scan(x_conv, dt, A, B_t, C_t)

        y = y * F.silu(z)
        return self.out_proj(y)

    def _selective_scan(self, x, dt, A, B, C):
        batch, T, D = x.shape
        N = self.d_state

        h = torch.zeros(batch, D, N, device=x.device, dtype=x.dtype)
        ys = []

        for t in range(T):
            dt_t = dt[:, t, :, None]
            dA = torch.exp(A[None] * dt_t)
            dB = dt_t * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            y_t = (h * C[:, t, None, :]).sum(-1)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class ResidualSSM(nn.Module):
    """Lightweight SSM for second-pass error correction."""

    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                 n_classes=234, n_windows=12, dropout=0.1, n_sites=20):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.site_emb = nn.Embedding(n_sites, 8)
        self.hour_emb = nn.Embedding(24, 8)
        self.meta_proj = nn.Linear(16, d_model)

        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm = nn.LayerNorm(d_model)
        self.ssm_drop = nn.Dropout(dropout)

        self.output_head = nn.Linear(d_model, n_classes)

        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass_scores, site_ids=None, hours=None):
        B, T, _ = emb.shape

        x = torch.cat([emb, first_pass_scores], dim=-1)
        h = self.input_proj(x)

        if site_ids is not None and hours is not None:
            site_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            hour_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([site_e, hour_e], dim=-1))
            h = h + meta.unsqueeze(1)

        h = h + self.pos_enc[:, :T, :]

        residual = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.ssm_merge(torch.cat([h_f, h_b], dim=-1))
        h = self.ssm_drop(h)
        h = self.ssm_norm(h + residual)

        return self.output_head(h)


# Load ResidualSSM state if available
residual_keys = [k for k in arr.files if k.startswith("residual_")]
if residual_keys:
    residual_state = {}
    for k in residual_keys:
        key_name = k.replace("residual_", "")
        residual_state[key_name] = torch.tensor(arr[k])

    res_model = ResidualSSM(
        d_input=1536,
        d_scores=N_CLASSES,
        d_model=64,
        d_state=8,
        n_classes=N_CLASSES,
        n_windows=N_WINDOWS,
        dropout=0.0,  # inference mode
        n_sites=20,
    ).to(DEVICE)
    res_model.load_state_dict(residual_state)
    res_model.eval()
    print(f"ResidualSSM loaded with {sum(p.numel() for p in res_model.parameters()):,} parameters")
else:
    res_model = None
    print("No ResidualSSM state found in artifacts")

# Site mapping for metadata
unique_sites = sorted(prior_tables["site_to_i"].keys()) if prior_tables.get("site_to_i") else []
site_to_idx = {s: i + 1 for i, s in enumerate(unique_sites)}
n_sites_max = 20

# =============================================================================
# LOAD PERCH
# =============================================================================
print("\nLoading Perch model...")

birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]

print("Perch loaded")

# =============================================================================
# UTILITIES
# =============================================================================
def smooth_cols(scores, cols, alpha=0.35):
    """Temporal smoothing for texture classes (moving average)."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s


def smooth_events(scores, cols, alpha=0.15):
    """Soft max-pool context for event birds (Aves)."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev_x, next_x))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s


def seq_features_1d(v):
    """Extract sequential features with std for temporal variance."""
    x = v.reshape(-1, N_WINDOWS)
    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v = np.repeat(x.max(axis=1), N_WINDOWS)
    std_v = np.repeat(x.std(axis=1), N_WINDOWS)
    return prev_v, next_v, mean_v, max_v, std_v


def build_class_features(emb, raw, prior, base):
    """Build features for probe prediction: embedding + 7 sequential + 3 interaction + std + 3 diff."""
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base)

    diff_mean = base - mean_base
    diff_prev = base - prev_base
    diff_next = base - next_base

    return np.concatenate([
        emb,
        raw[:, None],
        prior[:, None],
        base[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        std_base[:, None],
        diff_mean[:, None],
        diff_prev[:, None],
        diff_next[:, None],
        (raw * prior)[:, None],
        (raw * base)[:, None],
        (prior * base)[:, None],
    ], axis=1).astype(np.float32)


def prior_logits(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables["site_to_i"].get(str(s), -1) for s in sites), dtype=np.int32, count=n)
    hour_idx = np.fromiter((tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours), dtype=np.int32, count=n)
    sh_idx = np.fromiter((tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)), dtype=np.int32, count=n)

    valid = hour_idx >= 0
    if valid.any():
        nh = tables["hour_n"][hour_idx[valid]][:, None]
        p[valid] = nh / (nh + 8.0) * tables["hour_p"][hour_idx[valid]] + (1.0 - nh / (nh + 8.0)) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables["site_n"][site_idx[valid]][:, None]
        p[valid] = ns / (ns + 8.0) * tables["site_p"][site_idx[valid]] + (1.0 - ns / (ns + 8.0)) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables["sh_n"][sh_idx[valid]][:, None]
        p[valid] = nsh / (nsh + 4.0) * tables["sh_p"][sh_idx[valid]] + (1.0 - nsh / (nsh + 4.0)) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


def fuse_scores(base, sites, hours, tables):
    scores = base.copy()
    prior = prior_logits(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += BEST_FUSION["lambda_event"] * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += BEST_FUSION["lambda_texture"] * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += BEST_FUSION["lambda_proxy_texture"] * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = BEST_FUSION["lambda_event"] * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = BEST_FUSION["lambda_texture"] * prior[:, idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols(scores, idx_active_texture, alpha=BEST_FUSION["smooth_texture"])
    scores = smooth_events(scores, idx_active_event, alpha=BEST_FUSION["smooth_event"])
    return scores.astype(np.float32), prior

# =============================================================================
# PERCH INFERENCE
# =============================================================================
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")


def parse_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": None, "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}


def read_audio(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y


def infer_perch(paths, verbose=True):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    iterator = range(0, n_files, BATCH_FILES)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + BATCH_FILES - 1) // BATCH_FILES, desc="Perch")

    for start in iterator:
        batch_paths = paths[start:start + BATCH_FILES]
        x = np.empty((len(batch_paths) * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_start = write_row

        for path in batch_paths:
            y = read_audio(path)
            x[write_row - batch_start:write_row - batch_start + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
            meta = parse_filename(path.name)
            row_ids[write_row:write_row + N_WINDOWS] = [f"{path.stem}_{t}" for t in range(5, 65, 5)]
            sites[write_row:write_row + N_WINDOWS] = meta["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])
            write_row += N_WINDOWS

        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs["label"].numpy().astype(np.float32)
        emb = outputs["embedding"].numpy().astype(np.float32)

        scores[batch_start:write_row, MAPPED_POS] = logits[:write_row - batch_start, MAPPED_BC_INDICES]
        embeddings[batch_start:write_row] = emb

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[batch_start:write_row, pos] = logits[:write_row - batch_start, bc_idx_arr].max(axis=1)

        del x, outputs, logits, emb
        gc.collect()

    return pd.DataFrame({"row_id": row_ids, "site": sites, "hour_utc": hours}), scores, embeddings

# =============================================================================
# RUN INFERENCE
# =============================================================================
print("\n" + "-" * 60)
print("Running Inference")
print("-" * 60)

test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))

if len(test_paths) == 0:
    print(f"No test files. Using {DRYRUN_N_FILES} train soundscapes.")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:DRYRUN_N_FILES]
else:
    print(f"Test files: {len(test_paths)}")

meta_test, scores_raw, emb_test = infer_perch(test_paths)

print(f"scores_raw: {scores_raw.shape}, emb_test: {emb_test.shape}")

# =============================================================================
# BUILD PREDICTIONS - PASS 1: Perch + Prior + MLP Probes
# =============================================================================
print("\nBuilding predictions (Pass 1: Perch + Prior + MLP)...")

# Fuse with priors
base_scores, prior_scores = fuse_scores(
    scores_raw, meta_test["site"].to_numpy(), meta_test["hour_utc"].to_numpy(), prior_tables
)

# Transform embeddings
if emb_scaler is not None:
    emb_scaled = emb_scaler.transform(emb_test)
    Z = emb_pca.transform(emb_scaled).astype(np.float32)
else:
    emb_scaled = (emb_test - scaler_mean) / scaler_scale
    Z = np.dot(emb_scaled - pca_mean, pca_components.T).astype(np.float32)

# Apply probes (LR + MLP ensemble)
first_pass_scores = base_scores.copy()
probe_classes = set(lr_probes.keys()) | set(mlp_probes.keys())

for cls_idx in tqdm(probe_classes, desc="Probes"):
    X = build_class_features(Z, scores_raw[:, cls_idx], prior_scores[:, cls_idx], base_scores[:, cls_idx])
    
    # Get predictions
    lr_pred = lr_probes[cls_idx].decision_function(X) if cls_idx in lr_probes else 0.0
    mlp_proba = mlp_probes[cls_idx].predict_proba(X)[:, 1] if cls_idx in mlp_probes else 0.5
    mlp_pred = np.log(np.clip(mlp_proba, 1e-7, 1 - 1e-7) / np.clip(1 - mlp_proba, 1e-7, 1 - 1e-7))
    
    # Ensemble: LR + MLP
    if cls_idx in lr_probes and cls_idx in mlp_probes:
        ensemble_pred = 0.5 * lr_pred + 0.5 * mlp_pred
    elif cls_idx in mlp_probes:
        ensemble_pred = mlp_pred
    else:
        ensemble_pred = lr_pred
    
    first_pass_scores[:, cls_idx] = (1.0 - PROBE_ALPHA) * base_scores[:, cls_idx] + PROBE_ALPHA * ensemble_pred

print(f"Pass 1 score range: {first_pass_scores.min():.4f} to {first_pass_scores.max():.4f}")

# =============================================================================
# PASS 2: ResidualSSM Correction
# =============================================================================
final_scores = first_pass_scores.copy()

if res_model is not None and CORRECTION_WEIGHT > 0:
    print("\nApplying ResidualSSM correction (Pass 2)...")
    
    # Reshape to file-level for ResidualSSM
    n_test_files = len(test_paths)
    emb_files = emb_test.reshape(n_test_files, N_WINDOWS, -1)
    first_pass_files = first_pass_scores.reshape(n_test_files, N_WINDOWS, -1)
    
    # Get site/hour IDs for each file
    sites_arr = meta_test["site"].to_numpy()
    hours_arr = meta_test["hour_utc"].to_numpy()
    
    file_site_ids = np.zeros(n_test_files, dtype=np.int64)
    file_hour_ids = np.zeros(n_test_files, dtype=np.int64)
    
    for fi in range(n_test_files):
        row_idx = fi * N_WINDOWS
        site = sites_arr[row_idx]
        hour = hours_arr[row_idx]
        file_site_ids[fi] = min(site_to_idx.get(str(site), 0), n_sites_max - 1)
        file_hour_ids[fi] = int(hour) % 24 if hour >= 0 else 0
    
    # Run ResidualSSM
    with torch.no_grad():
        emb_t = torch.tensor(emb_files, dtype=torch.float32, device=DEVICE)
        first_pass_t = torch.tensor(first_pass_files, dtype=torch.float32, device=DEVICE)
        site_t = torch.tensor(file_site_ids, dtype=torch.long, device=DEVICE)
        hour_t = torch.tensor(file_hour_ids, dtype=torch.long, device=DEVICE)
        
        corrections = res_model(emb_t, first_pass_t, site_ids=site_t, hours=hour_t)
        corrections_np = corrections.cpu().numpy()
    
    # Apply correction
    corrected = first_pass_files + CORRECTION_WEIGHT * corrections_np
    final_scores = corrected.reshape(-1, N_CLASSES).astype(np.float32)
    
    print(f"Correction applied: weight={CORRECTION_WEIGHT}")
    print(f"Correction range: {corrections_np.min():.4f} to {corrections_np.max():.4f}")
else:
    print("\nResidualSSM skipped (not available or weight=0)")

print(f"Final score range: {final_scores.min():.4f} to {final_scores.max():.4f}")

# =============================================================================
# SUBMISSION
# =============================================================================
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))


final_probs = sigmoid(final_scores / TEMPERATURE)

submission = pd.DataFrame(final_probs, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", meta_test["row_id"].values)
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)

# Validate
expected_rows = len(test_paths) * N_WINDOWS
assert len(submission) == expected_rows
assert submission.columns.tolist() == ["row_id"] + PRIMARY_LABELS
assert not submission.isna().any().any()

submission.to_csv("submission.csv", index=False)

print("\n" + "=" * 60)
print("Inference Complete!")
print("=" * 60)
print(f"Submission shape: {submission.shape}")
print(submission.iloc[:3, :8])